In [164]:
import torch
torch.set_default_dtype(torch.float64)
from cpu_cb import E8P12_codebook
from itertools import product

In [165]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    b = torch.tensor([1,1,-1,-1]).to(x.dtype)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1) + (4,) + (1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k) / (2**k)

In [166]:
def calculate_k(n):
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    return k

In [167]:
def hb_transform_loop(x, k): # hb_transform without the end permutate and reshape
    (m,n) = x.shape
    assert 4**k == n
    b = torch.tensor([1, 1, -1, -1]).to(x.dtype)
    x = x.reshape((m,) + (4,) * k)
    for i in range(k):
        x = x.flip(1 + i) + x * b.view((1,) * (i + 1) + (4,) + (1,) * (k - 1 - i))
    # return x.reshape((m,) + (2, 2) * k) / (2**k)
    return x.reshape((m,) + (2, 2) * k)

In [168]:
def hb_transform_reshape(x, k): # given original hb_transform function shape go to the hb_transform_loop
    m = x.shape[0]
    x = x.reshape((m,) + (2,) * (2 * k))
    forward_perm = [0] + [2 * i + 1 for i in range(k)] + [2 * i + 2 for i in range(k)]
    inverse_perm = [0] * (2 * k + 1)
    for i, p in enumerate(forward_perm):
        inverse_perm[p] = i 
    x = x.permute(inverse_perm)
    return x.reshape((m,) + (4,) * k)

In [169]:
n = 64
k = calculate_k(n*n)
W = torch.rand(n, n)
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(1, n*n)
norm = 1/n
coeffs = coeffs / norm
hatW = hb_transform(coeffs).reshape(n, n)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 1.8375725281923241e-16


In [170]:
cliques = [
    [0, 2, 11, 25, 33, 39, 47, 57],
    [1, 3, 10, 24, 32, 38, 46, 56],
    [4, 6, 15, 29, 35, 37, 43, 61],
    [5, 7, 14, 28, 34, 36, 42, 60],
    [12, 18, 20, 26, 44, 53, 55, 62],
    [13, 19, 21, 27, 45, 52, 54, 63],
    [8, 16, 22, 30, 40, 49, 51, 58],
    [9, 17, 23, 31, 41, 48, 50, 59]
]
cb = E8P12_codebook()

In [171]:
def generate_cliques(n, cliques_8):
    num_cliques = n*n // 64
    scaled_cliques = []
    offsets = []
    for i in range(num_cliques):
        offsets.append(i * 64)
    for clique in cliques_8:
        for offset in offsets:
            new_clique = []
            for mat in clique:
                new_clique.append(mat + offset)
            scaled_cliques.append(new_clique)            
    return scaled_cliques

In [172]:
cliques_16 = generate_cliques(16, cliques)
mats = hb_transform(torch.eye(256))
for clique in cliques_16:
    for i in clique:
        for j in clique:
            if i != j:
                torch.testing.assert_close(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T))

In [173]:
n = 32
base_cliques = torch.tensor(cliques, dtype=torch.long, device=coeffs.device)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long, device=coeffs.device).view(-1, 1, 1) * 64
cliques_32= offsets + base_cliques.unsqueeze(0)
cliques_32 = cliques_32.reshape(-1, 8)
mats = hb_transform(torch.eye(32*32))
for clique in cliques_32:
    for i in clique:
        for j in clique:
            if i != j:
                torch.testing.assert_close(mats[i] @ mats[j].T, -1 * (mats[j] @ mats[i].T))

In [174]:
n = 256
base_cliques = torch.tensor(cliques, dtype=torch.long, device=coeffs.device)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long, device=coeffs.device).view(-1, 1, 1) * 64
cliques_16 = offsets + base_cliques.unsqueeze(0)
cliques_16 = cliques_16.reshape(-1, 8)

In [175]:
n = 64
base_cliques = torch.tensor(cliques, dtype=torch.long, device=coeffs.device)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long, device=coeffs.device).view(-1, 1, 1) * 64
cliques_64 = offsets + base_cliques.unsqueeze(0)
cliques_64 = cliques_64.reshape(-1, 8)
k = calculate_k(n*n)
W = torch.rand(n, n)
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(1, n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = torch.zeros_like(coeffs)
for clique in cliques_64:
    target = coeffs[:, clique]
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hat_coeffs = hat_coeffs.reshape(4096)
hatW = hb_transform(hat_coeffs).reshape(n, n) * Wscale
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.5971399658537105


In [156]:
n = 64
k = calculate_k(n*n)
base_cliques = torch.tensor(cliques, dtype=torch.long, device=coeffs.device)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long, device=coeffs.device).view(-1, 1, 1) * 64
cliques_8 = offsets + base_cliques.unsqueeze(0)
cliques_8 = cliques_8.reshape(-1, 8)
mats = hb_transform(torch.eye(n*n))
mats_flat = mats.reshape(n*n, n*n)
W = torch.randn(n, n)
A = torch.randn(n * n, n * n)
H = (A + A.T) / 2
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(1, n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = torch.zeros_like(coeffs)
for i in range(cliques_8.shape[0]):
    clique = cliques_8[i]
    target = coeffs[:, clique]
    if i != 0:
        clique_corrections = torch.zeros(1, 8)
        mats_i = mats_flat[clique]
        for j in range(i):
            clique_j = cliques_8[j]
            prev_errs = coeffs[:, clique_j] - hat_coeffs[:, clique_j]
            mats_j = mats_flat[clique_j]
            correction_trace = (mats_i @ H_sqrt @ mats_j.T) / tr_H_sqrt / (2**k)**2
            clique_corrections += prev_errs @ correction_trace.T
        target += clique_corrections
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hat_coeffs = hat_coeffs.reshape(4096) * Wscale
hatW = hb_transform(hat_coeffs).reshape(n, n)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.30058981804555923


In [204]:
n = 256
k = calculate_k(n*n)
base_cliques = torch.tensor(cliques, dtype=torch.long, device=coeffs.device)
num_cliques = n*n // 64
offsets = torch.arange(num_cliques, dtype=torch.long, device=coeffs.device).view(-1, 1, 1) * 64
cliques_8 = offsets + base_cliques.unsqueeze(0)
W = torch.randn(n, n)
A = torch.randn(n, n)
H = (A + A.T) / 2
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(1, n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = torch.zeros_like(coeffs)
row_i = torch.arange(n)
for group in range(cliques_8.shape[0]):
    perms, signs = [], []
    for member in range(64):
        flat_i = group * 64 + member
        ps_coeffs = torch.zeros(1, n * n)
        ps_coeffs[0, flat_i] = 1.0
        B = hb_transform(ps_coeffs).reshape(n, n) * (2**k)
        B_bool = (B != 0) # true if 1 and false ow
        perm = B_bool.double().argmax(dim=1) # to find where the 1 is
        # sign = B[torch.arange(B.shape[0]), perm] # extract signs at location in perm
        sign = B[row_i, perm] # extract signs at location in perm
        perms.append(perm.long())
        signs.append(sign)
    perms = torch.stack(perms)
    signs = torch.stack(signs)
    start = group * 64
    end = start + 64
    group_target = coeffs[0, start:end]
    group_hat = torch.zeros(64, dtype=coeffs.dtype)
    for ci, clique in enumerate(cliques_8[group]):
        target = group_target[clique%64].clone()
        if ci > 0:
            correction = torch.zeros(8, dtype=coeffs.dtype)
            p_i = perms[clique%64]
            s_i = signs[clique%64]
            for cp in range(ci):
                prior_clique = cliques_8[group][cp]
                prev_err = group_target[prior_clique%64] - group_hat[prior_clique%64]
                p_j = perms[prior_clique%64]
                s_j = signs[prior_clique%64]                
                p_j_t = torch.zeros_like(p_j)
                p_j_t.scatter_(1, p_j, row_i.expand(8, -1))                
                s_j_t = torch.zeros_like(s_j)
                s_j_t.scatter_(1, p_j, s_j)
                p_i_exp = p_i.unsqueeze(1).expand(-1, 8, -1) # (8, 8, n)
                s_i_exp = s_i.unsqueeze(1).expand(-1, 8, -1) # (8, 8, n)
                p_j_t_exp = p_j_t.unsqueeze(0).expand(8, -1, -1) # (8, 8, n)
                s_j_t_exp = s_j_t.unsqueeze(0).expand(8, -1, -1) # (8, 8, n)
                pi_times_pj_t = torch.gather(p_j_t_exp, 2, p_i_exp)
                si_times_sj_t = s_i * torch.gather(s_j_t_exp, 2, p_i_exp)
                H_curr = H[row_i.view(1, 1, -1), pi_times_pj_t] # (8, 8, n)
                cross = (H_curr * si_times_sj_t).sum(dim=2) / tr_H_sqrt / (2**k)**2 # [8, 8]
                # cross = torch.zeros(8, 8, dtype=coeffs.dtype)
                # for i, mi in enumerate(clique):
                #     for j, mj in enumerate(prior_clique):
                #         pj_t = torch.zeros_like(perms[mj%64]) # need to get transpose of second because tr(H B_i B_j^T)
                #         pj_t[perms[mj%64]] = torch.arange(perms[mj%64].shape[0])
                #         sj_t = torch.zeros_like(signs[mj%64])
                #         sj_t[perms[mj%64]] = signs[mj%64]
                #         pi_times_pj_t = pj_t[perms[mi%64]] # multipliying B_i and B_j^T
                #         si_times_sj_t = signs[mi%64] * sj_t[perms[mi%64]]
                #         cross[i, j] = (H[torch.arange(H.shape[0]), pi_times_pj_t] * si_times_sj_t).sum() / tr_H_sqrt / (2**k)**2
                correction += cross @ prev_err
            target += correction
        hat_clique, _ = cb.quantize(target.unsqueeze(0))
        group_hat[clique%64] = hat_clique.squeeze(0)
    hat_coeffs[0, start:end] = group_hat
hat_coeffs_flat = hat_coeffs.reshape(1, n * n)
hatW = hb_transform(hat_coeffs_flat).reshape(n, n) * Wscale
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.3009656631204808


In [13]:
def mats_tensor_product(n):
    rep = int(torch.log2(torch.tensor(n // 8)))
    A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
    B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
    mats = []
    for i in product(range(4), repeat=rep):
        for j in range(64):
            cands = [A2[k] for k in i] + [B8[j]]
            M = cands[0]
            for mat in cands[1:]:
                M = torch.kron(M, mat)
            mats.append(M)
    return torch.stack(mats) / mats[0].norm()

In [14]:
A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
n = 64
cb = E8P12_codebook()
# mats = []
# for i in range(4):
#     for j in range(4):
#         for k in range(4):
#             for l in range(64):
#                 mats.append(torch.kron(torch.kron(torch.kron(A2[i], A2[j]), A2[k]), B8[l]))
# mats = torch.stack(mats) / mats[0].norm() # (4096, 64, 64)
mats = mats_tensor_product(n)
W = torch.randn(n, n)
coeffs = (mats * W).sum(dim=(-2, -1)).reshape(64, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in cliques:
    target = coeffs[:, clique]
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hat_coeffs = hat_coeffs.reshape(4096)
hatW = (mats * hat_coeffs.view(4096, 1, 1)).sum(dim=0)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.3061708439533606


In [15]:
A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
n = 128
cb = E8P12_codebook()
mats = mats_tensor_product(n)
W = torch.randn(n, n)
coeffs = (mats * W).sum(dim=(-2, -1)).reshape((n * n) // 64, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in cliques:
    target = coeffs[:, clique]
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hat_coeffs = hat_coeffs.reshape(n * n)
hatW = (mats * hat_coeffs.view(n * n, 1, 1)).sum(dim=0)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.30254158096785516


In [16]:
A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
n = 64
cb = E8P12_codebook()
mats = mats_tensor_product(n)
W = torch.randn(n, n)
A = torch.randn(n * n, n * n)
H = A.T @ A + n * n * torch.eye(n * n)
mats_flat = mats.reshape(4096, 4096)
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
H_sqrt_coeffs = mats_flat @ H_sqrt @ mats_flat.T # (4096, 4096)
mats_flat = mats.reshape(4096, 4096)
flat_coeffs = mats_flat @ W.reshape(4096)
coeffs = flat_coeffs.reshape(64, 64) # (64, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in range(8):
    target = coeffs[:, cliques[clique]] # (64, 8)
    curr_clique_inds_flat = [i * 64 + j for j in cliques[clique] for i in range(64)]
    if clique != 0:
        correction = torch.zeros(64, 8)
        for i in range(clique):
            prev_clique_inds_flat = [a * 64 + b for b in cliques[i] for a in range(64)]
            prev_errors = (coeffs[:, cliques[i]] - hat_coeffs[:, cliques[i]]) # (64, 8)
            prev_errors_flat = prev_errors.reshape(-1) # (512,)
            H_sqrt_coeffs_part = H_sqrt_coeffs[curr_clique_inds_flat, :][:, prev_clique_inds_flat] # (512, 512)
            correction_flat = (H_sqrt_coeffs_part @ prev_errors_flat) / tr_H_sqrt # (512,)
            correction += correction_flat.reshape(64, 8)
        target += correction
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, cliques[clique]] = hat_clique
hatW_flat = mats_flat.T @ hat_coeffs.reshape(4096)
hatW = hatW_flat.reshape(64, 64) # (64, 64)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.3015584256724117
